# Model comparison against a trivial baseline

Scores the two models from `2_dim/2_dim_pipeline.ipynb` against a baseline with no model at all, on the same 15 test winters, using that pipeline's saved per-winter results and its definitions of B, SB and SBE. Writes `results/model_comparison.csv`.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

TRAIN_YEARS = range(1957, 1987)   # winter = December of this year to February of the next
TEST_YEARS = range(1987, 2002)
MEASURES = ["LIB", "SB", "SBE"]
NDAYS = 5

saved = np.load("../2_dim/notebook_output/winter_freqs_test.npz")
assert list(saved["years"]) == list(TEST_YEARS)
models = {m: {k: saved[f"{m}_{k}"] for k in MEASURES} for m in ("eigenbasis", "mhw")}   # (winters, longitudes)
data = {k: saved[f"data_{k}"] for k in MEASURES}

## Trivial baseline

Predict each longitude's frequency as its average over the training winters. It uses no model and no information from the test winters. Frequencies follow the pipeline: SBE is the share of 5-day windows with SB on all 5 days. `episodes/` holds the same pipeline's daily flags, which is checked against its saved test values first.

In [2]:
def winter_frequencies(year):
    lib = xr.open_dataset(f"../episodes/LIB_{year}.nc").LIB.values.astype(bool)
    sb = xr.open_dataset(f"../episodes/SB_{year}.nc").SB.values.astype(bool)
    windows = np.stack([sb[d + 1: d + 1 + NDAYS] for d in range(len(sb) - NDAYS)])   # same windows as the pipeline
    return {"LIB": lib.mean(0), "SB": sb.mean(0), "SBE": windows.all(axis=1).mean(0)}

check = [winter_frequencies(y) for y in TEST_YEARS]
for k in MEASURES:
    print(f"{k}: largest difference from the pipeline's saved test values: {np.abs(np.array([c[k] for c in check]) - data[k]).max():.2e}")

train = [winter_frequencies(y) for y in TRAIN_YEARS]
baseline = {k: np.mean([t[k] for t in train], axis=0) for k in MEASURES}   # fixed curves, one value per longitude

sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found


sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found


LIB: largest difference from the pipeline's saved test values: 0.00e+00
SB: largest difference from the pipeline's saved test values: 0.00e+00
SBE: largest difference from the pipeline's saved test values: 0.00e+00


sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found


sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found


sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found


sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found
sh: line 1: getfattr: command not found


## Scores with bootstrap intervals

RMSE between mean curves over the test winters. The 15 winters are resampled with replacement 10,000 times; each model's curve uses the same resampled winters as the data, and the baseline stays fixed. An interval for a difference that excludes 0 means a clear difference.

In [3]:
def rmse(a, b):
    return np.sqrt(((a - b) ** 2).mean(axis=-1))

N_BOOT = 10_000
rng = np.random.default_rng(0)
picks = rng.integers(0, len(TEST_YEARS), size=(N_BOOT, len(TEST_YEARS)))

rows, boots = [], {}
for k in MEASURES:
    d = data[k]
    se = d.std(0, ddof=1) / np.sqrt(len(d))
    curves = {"trivial (training-winter frequency)": baseline[k], "MHW-style AR(1)": models["mhw"][k].mean(0), "eigenbasis AR(1)": models["eigenbasis"][k].mean(0)}
    d_boot = d[picks].mean(1)
    b = {"trivial (training-winter frequency)": rmse(baseline[k][None], d_boot),
         "MHW-style AR(1)": rmse(models["mhw"][k][picks].mean(1), d_boot),
         "eigenbasis AR(1)": rmse(models["eigenbasis"][k][picks].mean(1), d_boot)}
    boots[k] = b
    for name, c in curves.items():
        lo, hi = np.percentile(b[name], [2.5, 97.5])
        rows.append(dict(split="train 1957-86, test 1987-2001", measure=k, model=name, rmse=rmse(c, d.mean(0)), rmse_low=lo, rmse_high=hi,
                         pct_longitudes_inside_data_band=100 * np.mean(np.abs(c - d.mean(0)) <= 2 * se), sector_330_90_mean=c[(np.arange(0, 360, 5) >= 330) | (np.arange(0, 360, 5) <= 90)].mean()))
comparison = pd.DataFrame(rows)
comparison.round(4).to_csv("../results/model_comparison.csv", index=False)
print(comparison.drop(columns="split").round(4).to_string(index=False))

measure                               model   rmse  rmse_low  rmse_high  pct_longitudes_inside_data_band  sector_330_90_mean
    LIB trivial (training-winter frequency) 0.0208    0.0161     0.0331                          73.6111              0.1624
    LIB                     MHW-style AR(1) 0.0128    0.0118     0.0219                          87.5000              0.1545
    LIB                    eigenbasis AR(1) 0.0132    0.0114     0.0222                          84.7222              0.1593
     SB trivial (training-winter frequency) 0.0321    0.0230     0.0506                          72.2222              0.2702
     SB                     MHW-style AR(1) 0.0145    0.0112     0.0305                         100.0000              0.2514
     SB                    eigenbasis AR(1) 0.0242    0.0163     0.0392                          95.8333              0.2828
    SBE trivial (training-winter frequency) 0.0123    0.0081     0.0211                          79.1667              0.0590


In [4]:
pairs = [("eigenbasis AR(1)", "MHW-style AR(1)"), ("eigenbasis AR(1)", "trivial (training-winter frequency)"), ("MHW-style AR(1)", "trivial (training-winter frequency)")]
rows = []
for k in MEASURES:
    for a, b in pairs:
        diff = boots[k][a] - boots[k][b]
        lo, hi = np.percentile(diff, [2.5, 97.5])
        rows.append(dict(measure=k, comparison=f"{a} minus {b}", low=lo, high=hi, clear_difference=(lo > 0) or (hi < 0)))
differences = pd.DataFrame(rows)
differences.round(4).to_csv("../results/model_comparison_rmse_differences.csv", index=False)
print(differences.round(4).to_string(index=False))

measure                                                 comparison     low    high  clear_difference
    LIB                     eigenbasis AR(1) minus MHW-style AR(1) -0.0021  0.0015             False
    LIB eigenbasis AR(1) minus trivial (training-winter frequency) -0.0126 -0.0031              True
    LIB  MHW-style AR(1) minus trivial (training-winter frequency) -0.0121 -0.0027              True
     SB                     eigenbasis AR(1) minus MHW-style AR(1) -0.0053  0.0157             False
     SB eigenbasis AR(1) minus trivial (training-winter frequency) -0.0195  0.0002             False
     SB  MHW-style AR(1) minus trivial (training-winter frequency) -0.0247 -0.0042              True
    SBE                     eigenbasis AR(1) minus MHW-style AR(1) -0.0119 -0.0006              True
    SBE eigenbasis AR(1) minus trivial (training-winter frequency) -0.0127  0.0032             False
    SBE  MHW-style AR(1) minus trivial (training-winter frequency) -0.0116  0.0142         